# Grafici commentati dei risultati principali

            Questo notebook raccoglie le viste piu utili per una lettura rapida: quote per classe, differenza tra peso numerico e peso economico, focus Italia e struttura settoriale.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from valore_aggiunto_imprese.analysis import (
    METRIC_LABELS,
    add_source_note,
    latest_common_year_with_values,
    latest_year_with_values,
    SIZE_ORDER_BUSINESS,
    SIZE_ORDER_OFFICIAL,
    add_share,
    enrich_business_demography,
    enrich_sbs,
    read_project_csv,
)

pd.options.display.max_columns = 80
pd.options.display.float_format = "{:,.2f}".format

## Base dati

In [ ]:
sbs = enrich_sbs(read_project_csv("data/processed/official_eurostat_sbs.csv"))
bd = enrich_business_demography(read_project_csv("data/processed/business_demography_distribution.csv"))
latest_year = latest_year_with_values(sbs, metric_code="AV_MEUR")
latest_bd_year = latest_year_with_values(bd, metric_code="V11910")

pd.DataFrame(
    [
        {"dataset": "Eurostat SBS", "righe": len(sbs), "ultimo_anno": latest_year},
        {"dataset": "Business Demography", "righe": len(bd), "ultimo_anno": latest_bd_year},
    ]
)

## Figura 1 - Quote di valore aggiunto

In [ ]:
va = (
    sbs.query("metric_code == 'AV_MEUR' and year == @latest_year")
    .dropna(subset=["value"])
    .groupby(["country_label", "size_label"], observed=True, as_index=False)["value"]
    .sum()
)
va["size_label"] = pd.Categorical(va["size_label"], SIZE_ORDER_OFFICIAL, ordered=True)
va_share = add_share(va, ["country_label"])

fig = px.bar(
    va_share.sort_values(["country_label", "size_label"]),
    x="country_label",
    y="share_pct",
    color="size_label",
    category_orders={"size_label": SIZE_ORDER_OFFICIAL},
    labels={"country_label": "Paese", "share_pct": "% valore aggiunto", "size_label": "Classe"},
    title=f"Valore aggiunto per classe dimensionale - {latest_year}",
)
add_source_note(fig, "Eurostat Structural Business Statistics")
fig

## Figura 2 - Italia nel tempo

In [ ]:
it_va = (
    sbs.query("metric_code == 'AV_MEUR' and country_code == 'IT'")
    .dropna(subset=["value"])
    .groupby(["year", "size_label"], observed=True, as_index=False)["value"]
    .sum()
)
it_va["size_label"] = pd.Categorical(it_va["size_label"], SIZE_ORDER_OFFICIAL, ordered=True)

fig = px.line(
    it_va.sort_values(["year", "size_label"]),
    x="year",
    y="value",
    color="size_label",
    markers=True,
    category_orders={"size_label": SIZE_ORDER_OFFICIAL},
    labels={"year": "Anno", "value": "Milioni di euro", "size_label": "Classe"},
    title="Italia: valore aggiunto per classe dimensionale",
)
add_source_note(fig, "Eurostat Structural Business Statistics")
fig

## Figura 3 - Micro-classi da Business Demography

In [ ]:
bd_micro = (
    bd.query("metric_code == 'V11910' and year == @latest_bd_year and size_label in @SIZE_ORDER_BUSINESS")
    .dropna(subset=["value"])
    .groupby(["country_label", "size_label"], observed=True, as_index=False)["value"]
    .sum()
)
bd_micro["size_label"] = pd.Categorical(bd_micro["size_label"], SIZE_ORDER_BUSINESS, ordered=True)
bd_micro_share = add_share(bd_micro, ["country_label"])

fig = px.bar(
    bd_micro_share.sort_values(["country_label", "size_label"]),
    x="country_label",
    y="share_pct",
    color="size_label",
    category_orders={"size_label": SIZE_ORDER_BUSINESS},
    labels={"country_label": "Paese", "share_pct": "% imprese fino a 9 dipendenti", "size_label": "Classe"},
    title=f"Micro-classi osservate in Business Demography - {latest_bd_year}",
)
add_source_note(fig, "Eurostat Business Demography")
fig

## Figura 4 - Settori e grandi imprese

In [ ]:
sector_va = (
    sbs.query("metric_code == 'AV_MEUR' and year == @latest_year")
    .dropna(subset=["value"])
    .groupby(["country_label", "sector_label", "size_label"], observed=True, as_index=False)["value"]
    .sum()
)
sector_share = add_share(sector_va, ["country_label", "sector_label"])
large_matrix = sector_share.query("size_label == '250+'").pivot_table(
    index="country_label",
    columns="sector_label",
    values="share_pct",
    aggfunc="sum",
    observed=True,
).round(1)

fig = px.imshow(
    large_matrix,
    aspect="auto",
    color_continuous_scale="Viridis",
    labels={"x": "Settore", "y": "Paese", "color": "% VA 250+"},
    title=f"Peso delle imprese 250+ nel valore aggiunto settoriale - {latest_year}",
)
add_source_note(fig, "Eurostat Structural Business Statistics")
fig

## Sintesi numerica

In [ ]:
(
    va_share.pivot_table(
        index="country_label",
        columns="size_label",
        values="share_pct",
        aggfunc="sum",
        observed=True,
    )
    .reindex(columns=SIZE_ORDER_OFFICIAL)
    .round(1)
    .sort_values("250+", ascending=False)
)